# Clean raw data

In [2]:
import pandas as pd
import numpy as np
# remove warnings
import warnings
warnings.filterwarnings("ignore")


# convert these
filenames = ['interleaved_incomplete',
             'block_incomplete',
             'block_complete',
             'block_complete_simple',
            ]

# to these
filenames2 = ['exp_1_interleaved_incomplete',
              'exp_2_block_incomplete',
              'exp_3_block_complete',
              'exp_4_block_complete_simple',
            ]


for f, f2 in zip(filenames, filenames2):
    df = pd.read_csv(f'data/behavior/raw/{f}.csv')

    df = df.rename(columns={
        'EXPID': 'sub', 'ID': 'prolific_id', 'EXP': 'group',
        'ELIC': 'elic', 'P1': 'p1', 'P2': 'p2', 'RTIME': 'rtime', 'OUT': 'out', 'CF_OUT': 'cfout', 'CHOICE': 'cho', 'CORRECT_CHOICE': 'corr', 'TEST': '10',
        'TRIAL': 'trial', 'COND': 'cond', 'CONT1': 'cont1', 'CONT2': 'cont2', 'SYML': '15', 'SYMR': '16', 'LR': '17', 'REW': 'rew', 'SESSION': 'sess', 
        'OP1': 'op1', 'OP2': 'op2', 'EV1': 'ev1', 'EV2': 'ev2', 'CATCH': 'catch', 'INV': '25', 'CTIME': '26', 'DIST': 'dist', 'PLOT': 'plot', 'DBTIME': 'dbtime'
    })

    df.drop([
             'prolific_id',
             'cont1', 'cont2',
             'dbtime', '10',
             '26', '15', '16',
             'dist', 'plot'], axis=1, inplace=True)

    df.rename(columns={'17': 'chose_right', 'elic': 'phase',
              'sub': 'sub_id', '25': 'reversed'}, inplace=True)
    df['chose_right'] = (df.chose_right > 0).astype(int)

    count = 0
    for sub in df.sub_id.unique():

        if np.isnan(sub):
            # drop nan sub
            df.drop(df[df.sub_id == sub].index, inplace=True)

        LE = (df.phase == -1) & (df.cond != -1) & (df.sub_id == sub)
        # count += sum(LE) == 120
        df.loc[LE, 'phase'] = 'LE'
        df.loc[LE, 'op1'] = 'E'
        df.loc[LE, 'op2'] = 'E'

        ES = (df.phase == 0) & (df.sub_id == sub) & (df.op2.isin(
            [0, -1])) & (df.op1.isin([1, -1, 0])) & (df.sub_id == sub)
        # assert len(LE[ES]) == 0
        # assert all(LE[ES] == False)
        # print(sum(ES))
        df.loc[ES, 'phase'] = 'ES'

        SP = (df.phase == 2) & (df.sub_id == sub)
        df.loc[SP, 'phase'] = 'SP'
        df.loc[SP & (df.catch == 0), 'op1'] = 'E'
        df.loc[SP & (df.catch == 1), 'op1'] = 'S'
        df.loc[SP, 'op2'] = np.NaN
        df.loc[SP, 'p2'] = np.NaN
        df.loc[SP, 'ev2'] = np.NaN
        df.loc[SP, 'cond'] = np.NaN

        df.loc[ES, 'op1'] = 'E'
        df.loc[ES, 'op2'] = 'S'
        df.loc[ES, 'cond'] = np.NaN

    df.loc[df.catch == -1, 'catch'] = 0
    df.loc[df.op2 == -1, 'op2'] = np.NaN
    df.loc[df.p2 == -1, 'op2'] = np.NaN
    print('Saving ', f2)
    df.rename(columns={'catch': 'catch_trial'}, inplace=True)
    df.to_csv(f'data/behavior/reformat/{f2}.csv', index_label='index')


filenames = [
             'block_complete_mixed',
             'block_complete_mixed_2s',
             'block_complete_mixed_2s_amb_heuristic',
             'block_complete_mixed_2s_amb_final',
             'incentivize',
             'block_complete_mixed_2s_nofixed',
             'nofixed_complete_feedback',
             'lottery_outcome'
]

filenames2 = [
              'exp_5_block_complete_mixed',
              'exp_6_block_complete_mixed_2s',
              'exp_7_block_complete_mixed_2s_deactivate',
              'exp_8_block_complete_mixed_2s_amb',
              'exp_9_incentives',
              'exp_10_nofixed',
              'exp_11_nofixed_feedback',
              'exp_12_lottery_outcome'
]


for f, f2 in zip(filenames, filenames2):

    df = pd.read_csv(f'data/behavior/raw/{f}.csv')

    df = df.rename(columns={
        'EXPID': 'sub', 'ID': 'prolific_id', 'EXP': 'group',
        'ELIC': 'elic', 'P1': 'p1', 'P2': 'p2', 'RTIME': 'rtime', 'OUT': 'out', 'CF_OUT': 'cfout', 'CHOICE': 'cho', 'CORRECT_CHOICE': 'corr', 'TEST': '10',
        'TRIAL': 'trial', 'COND': 'cond', 'CONT1': 'cont1', 'CONT2': 'cont2', 'SYML': '15', 'SYMR': '16', 'LR': '17', 'REW': 'rew', 'SESSION': 'sess', 
        'OP1': 'op1', 'OP2': 'op2', 'EV1': 'ev1', 'EV2': 'ev2', 'CATCH': 'catch', 'INV': '25', 'CTIME': '26', 'DIST': 'dist', 'PLOT': 'plot', 'DBTIME': 'dbtime'
    })
    df.drop([
             'prolific_id',
            #  'cont1', 'cont2',
             'dbtime', '10',
             '26', '15', '16',
             'dist', 'plot'], axis=1, inplace=True)

    df.rename(columns={'17': 'chose_right', 'elic': 'phase',
              'sub': 'sub_id', '25': 'reversed'}, inplace=True)
    df['chose_right'] = (df.chose_right > 0).astype(int)

    

    for sub in df.sub_id.unique():
        if np.isnan(sub):
            # drop nan sub
            df.drop(df[df.sub_id == sub].index, inplace=True)

        LE = (df.phase == -1) & (df.op1==1) & (df.op2==1) & (df.sub_id == sub)
        # count += sum(LE) == 120
        df.loc[LE, 'phase'] = 'LE'
        df.loc[LE, 'op1'] = 'E'
        df.loc[LE, 'op2'] = 'E'

        ES = (df.phase == 0) & (df.sub_id == sub) & (df.op2.isin(
            [0] )) & (df.op1.isin([0, 1])) & (df.sub_id == sub)
        # assert len(LE[ES]) == 0
        # assert all(LE[ES] == False)
        # print(sum(ES))
        df.loc[ES, 'phase'] = 'ES'

        SP = (df.phase == 2) & (df.sub_id == sub)
        df.loc[SP, 'phase'] = 'SP'
        df.loc[SP & (df.catch == 0), 'op1'] = 'E'
        df.loc[SP & (df.catch == 1), 'op1'] = 'S'
        df.loc[SP, 'op2'] = np.NaN
        df.loc[SP, 'p2'] = np.NaN
        df.loc[SP, 'ev2'] = np.NaN
        df.loc[SP, 'cond'] = np.NaN
        df.loc[SP, 'catch'] = 0

        df.loc[ES, 'op1'] = 'E'
        df.loc[ES, 'op2'] = 'S'
        df.loc[ES, 'cond'] = np.NaN

        EE = (df.phase == 0) & (df.op2 == 1) & (df.sub_id == sub)
        assert all((EE[ES] == False))
        df.loc[EE, 'phase'] = 'EE'
        df.loc[EE, 'op1'] = 'E'
        df.loc[EE, 'op2'] = 'E'
        df.loc[EE, 'cond'] = np.NaN
            # print(sum(EE)) 

        if 'amb' in f:
            EA = (df.phase == 0) & (df.op2 == 2) & (
                df.op1 == 1) & (df.sub_id == sub)
            df.loc[EA, 'phase'] = 'EA'

            df.loc[EA, 'op1'] = 'E'
            df.loc[EA, 'op2'] = 'A'
            df.loc[EA, 'cond'] = np.NaN

            SA = (df.phase == 0) & (df.op2 == 2) & (
                df.op1 == 0) & (df.sub_id == sub)

            df.loc[SA, 'phase'] = 'SA'
            df.loc[SA, 'op1'] = 'S'
            df.loc[SA, 'op2'] = 'A'
            df.loc[SA, 'cond'] = np.NaN
        
        
    if f2 == 'exp_12_lottery_outcome':
        # here i don't know why it was reversed
        df.loc[df.group == 'EvOutcome', 'group'] = 'LotteryOutcome2'
        df.loc[df.group == 'EvOutcome2', 'group'] = 'LotteryOutcome1'
        
        

    df.loc[df.catch == -1, 'catch'] = 0
    df.loc[df.op2 == -1, 'op2'] = np.NaN
    df.loc[df.p2 == -1, 'op2'] = np.NaN
    df.rename(columns={'catch': 'catch_trial'}, inplace=True)
    print('Saving ', f2)
    df.to_csv(f'data/behavior/reformat/{f2}.csv', index_label='index')


Saving  exp_1_interleaved_incomplete
Saving  exp_2_block_incomplete
Saving  exp_3_block_complete
Saving  exp_4_block_complete_simple
Saving  exp_5_block_complete_mixed
Saving  exp_6_block_complete_mixed_2s
Saving  exp_7_block_complete_mixed_2s_deactivate
Saving  exp_8_block_complete_mixed_2s_amb
Saving  exp_9_incentives
Saving  exp_10_nofixed
Saving  exp_11_nofixed_feedback
Saving  exp_12_lottery_outcome


# RECODE WITHIN SUBJECTS SESSIONS 
* NoFixed1: nofixed learning in session 1 (0 in the csv), fixed learning in session 2 (1 in the csv) 
* Nofixed2: the opposite
* Feedback1: display feedback in session 1, do not in session 2
* Feedback2: reversed
* LotteryOutcome1: show a lottery as feedback in session 1; show integers (-1, 1; as before) in session 2
* LotteryOutcome2: reversed

We recode this to make analysis easier -> session 1 is wo/ the manipulation, session 2 is with


In [3]:
filenames = [
    'exp_10_nofixed',
    'exp_11_nofixed_feedback',
    'exp_12_lottery_outcome'
]
groups = [
    'NoFixed1',
    'Feedback1',
    'LotteryOutcome1',
]

for i, f in enumerate(filenames):

    df = pd.read_csv(f'data/behavior/reformat/{f}.csv')
    mask = (df.group == groups[i]) & (df.sess >= 0)

    # reverse session (from 0 to 1 or 1 to 0)
    df.loc[mask, 'sess'] = 1 - df.loc[mask, 'sess']

    df.to_csv(f'data/behavior/reformat/{f}_within_recoded.csv', index_label='index')    